# Stage 4 ? Human Gold Standard & Gemini-Assisted ABSA Labeling

BrewScope ? 26 Oktober 2026 ? taxonomy `0.1.0-draft`.

Notebook ini menyiapkan dan memvalidasi **infrastruktur Stage 4A**. Run All memakai
fixture **sintetis/MockProvider**, tanpa API key, tanpa jaringan, dan tanpa memberi
label data nyata. Formulir pilot manusia berasal dari 250 kandidat asli, masih kosong.

## 1. Research Overview

Task A: presence enam aspek (multi-label 0?6). Task B: positive/neutral/negative
pada aspek yang disebut. Mixed disimpan dengan dua evidence dan sentiment null;
uncertain merupakan workflow. Rating bukan ground truth.

Pipeline reusable berada di `src/brewscope/annotation`, sesuai struktur repo.
[Protokol lengkap](../docs/research/stage4_annotation_protocol.md) dan
[keputusan metodologi](../docs/research/stage4_methodology_decisions.md).

In [1]:
from pathlib import Path
import copy
import importlib.metadata
import json
import sys
import tempfile
import pandas as pd
import yaml
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/aspect_taxonomy.yaml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from brewscope.annotation.agreement import inter_annotator_agreement
from brewscope.annotation.evaluation import evaluate_predictions
from brewscope.annotation.export import assert_no_leakage
from brewscope.annotation.human import load_gold, prepare_pilot
from brewscope.annotation.pipeline import AnnotationPipeline, checkpoint_records, quality_queue
from brewscope.annotation.providers import MockProvider, api_key
from brewscope.annotation.runtime import (check_live_gates, frozen_prompt, load_runtime,
    source_reviews, synthetic_suite)
from brewscope.annotation.storage import atomic_json, digest, file_hash, read_json
from brewscope.annotation.validation import ASPECTS, AnnotationError, AnnotationValidator, parse_inference

CONFIG = load_runtime(ROOT)
VALIDATOR = AnnotationValidator(ROOT / 'schemas/annotation_schema.json')
EXECUTE_LIVE = False  # Run All tidak memiliki jalur panggilan cloud.
print({'stage': 4, 'live_enabled': EXECUTE_LIVE, 'mode': 'SYNTHETIC_ONLY'})

{'stage': 4, 'live_enabled': False, 'mode': 'SYNTHETIC_ONLY'}


## 2. Stage 3 Taxonomy Recap

Enam kategori aktif tetap sama. `location_accessibility` provisional dan tidak
masuk schema produksi. Schema canonical Stage 3 tidak diubah; schema inference
lebih sederhana dikonversi dan divalidasi lengkap secara lokal.

In [2]:
taxonomy = yaml.safe_load((ROOT / 'configs/aspect_taxonomy.yaml').read_text(encoding='utf-8'))
display(pd.DataFrame([{'aspect_id': a['id'], 'active': a['active'],
    'definition': a['operational_definition']} for a in taxonomy['aspects']]))
assert tuple(a['id'] for a in taxonomy['aspects'] if a['active']) == ASPECTS
assert taxonomy['taxonomy_version'] == VALIDATOR.version

,aspect_id,active,definition
0,product_quality,True,"Kualitas intrinsik makanan/minuman, rasa, kese..."
1,service,True,"Interaksi dan proses layanan staf: keramahan, ..."
2,price_value,True,"Evaluasi harga, keterjangkauan, promosi, dan k..."
3,ambience,True,"Pengalaman suasana dan kenyamanan: interior, k..."
4,facilities,True,"Keberadaan, kapasitas, fungsi, dan kemudahan m..."
5,cleanliness,True,"Keadaan bersih/higienis ruang, meja, toilet, p..."
6,location_accessibility,False,"Kenyamanan posisi geografis, kemudahan menemuk..."


## 3. Dataset & Usage Readiness

Brief Stage 4 memberi deklarasi **OWNER_ATTESTED** untuk penelitian dan AI
external. Deklarasi ini berbeda dari lisensi pihak sumber terverifikasi. Seluruh
40 lokasi tetap UNVERIFIED; pilot terkontrol boleh disiapkan. Verifikasi sumber
serta cakupan administratif diperlukan sebelum klaim/publikasi dan training akhir.

Raw tidak diubah atau dihapus. PII pola ditahan; pemeriksaan manusia terhadap
informasi pribadi kontekstual wajib sebelum batch nyata dikirim ke Gemini.

In [3]:
reviews, stage3_manifest = source_reviews(ROOT)
raw_hash_before = file_hash(ROOT / 'data/raw/google_maps_reviews_jatinangor_raw.csv')
print({'raw_reviews': 3593, 'eligible_technical': len(reviews),
    'shops': len({r['coffee_shop_id'] for r in reviews}),
    'geography': sorted({r['geography_status'] for r in reviews}),
    'authorization': CONFIG['authorization']['status'],
    'independent_source_permission_verified': False,
    'actual_labels_created': False})
assert len(reviews) == 3582
# Tidak menampilkan teks review nyata, rating, atau credential.

{'raw_reviews': 3593, 'eligible_technical': 3582, 'shops': 40, 'geography': ['UNVERIFIED'], 'authorization': 'OWNER_ATTESTED', 'independent_source_permission_verified': False, 'actual_labels_created': False}


## 4. Gemini API Setup

SDK resmi `google-genai` dipasang melalui extra `annotation`. API key lokal:
`GEMINI_API_KEY` di `.env` atau environment variable; jangan masukkan key ke chat
atau notebook. Buat key melalui [AI Studio](https://aistudio.google.com/api-keys).

[SDK](https://googleapis.github.io/python-genai/) mendukung structured response;
[rate limits](https://ai.google.dev/gemini-api/docs/rate-limits) aktual harus dilihat
pada project AI Studio. API lookup model tidak membuktikan billing Free Tier.

In [4]:
setup_status = {'google-genai': importlib.metadata.version('google-genai'),
    'api_key_present': bool(api_key(ROOT)), 'billing_tier': CONFIG['project']['billing_tier'],
    'rpm_configured': CONFIG['project']['rpm'], 'daily_requests_configured': CONFIG['project']['daily_requests'],
    'cloud_generation_in_this_notebook': 0}
print(setup_status)
# Key tidak pernah dicetak. Tidak menginisialisasi Gemini client dari Run All.

{'google-genai': '2.33.0', 'api_key_present': False, 'billing_tier': 'UNKNOWN', 'rpm_configured': None, 'daily_requests_configured': None, 'cloud_generation_in_this_notebook': 0}


## 5. Provider Configuration

GeminiProvider utama, OllamaProvider opsional, MockProvider untuk fixture.
Kandidat awal `gemini-3.5-flash-lite` memiliki structured output dan Free Tier
menurut dokumentasi publik yang diperiksa 26 Oktober 2026. Akses project aktual
belum diverifikasi. Kandidat 2.5 dari brief tidak diasumsikan bisa diakses project
baru; dokumentasi kini membatasi akses pengguna baru.

[Model](https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite),
[pricing](https://ai.google.dev/gemini-api/docs/pricing),
[ketentuan pemrosesan](https://ai.google.dev/gemini-api/terms).
Model awal belum dipilih berdasarkan akurasi gold.

In [5]:
synthetic_reviews, synthetic_responses = synthetic_suite(ROOT)
print({'configured_primary_provider': CONFIG['provider'], 'candidate_model': CONFIG['model'],
    'notebook_provider': 'mock', 'synthetic_reviews': len(synthetic_reviews),
    'paid_fallback': False, 'quota_status': 'PROJECT_NOT_VERIFIED'})

{'configured_primary_provider': 'gemini', 'candidate_model': 'gemini-3.5-flash-lite', 'notebook_provider': 'mock', 'synthetic_reviews': 17, 'paid_fallback': False, 'quota_status': 'PROJECT_NOT_VERIFIED'}


## 6. Human Pilot Design

Gunakan tepat 250 kandidat Stage 3. Split kelompok deterministik: 50 kalibrasi,
200 independen (150 development + 50 locked validation). Teks identik/sangat mirip
dikelompokkan di seluruh eligible corpus; tidak dihapus. Near-match konservatif
memerlukan shared 5-token shingle, ?8 token, Jaccard ?.85, SequenceMatcher ?.92.
Parafrasa semantik jauh tetap memerlukan review manusia.

A/B menilai independen, C adjudikasi, D QA. Paket A/B tidak memuat rating,
keyword hints, label AI, hasil orang lain atau identitas locked split. Slot
annotator tidak menandakan manusia sudah mengisi. Formulir draft bukan label.

In [6]:
pilot_preparation = prepare_pilot(ROOT, VALIDATOR)
print(pilot_preparation)
pilot_manifest = read_json(ROOT / 'data/interim/human_annotations/pilot_manifest.json')
assignment_frame = pd.DataFrame(pilot_manifest['assignments'])
display(assignment_frame.groupby('split').agg(reviews=('review_id','count'),
    shops=('coffee_shop_id','nunique')))
assert len(assignment_frame) == 250
assert assignment_frame.groupby('duplicate_group')['split'].nunique().max() == 1

{'reviews': 250, 'shops': 40, 'split_counts': {'development': 150, 'calibration': 50, 'locked_validation': 50}, 'template_decisions_per_annotator': 1500, 'state': 'UNANNOTATED', 'source_sha256': '7b87a53110041da74e976a6c9cb121c1dea7d73be70daeab58b2c07bef775719'}


,reviews,shops
split,,
calibration,50,31
development,150,39
locked_validation,50,27


## 7. Annotation Schema Validation

Kontrak: tepat enam keputusan unik per review, identitas/versi sesuai sumber,
presence?sentiment konsisten, mixed positif+negatif dan evidence verbatim.
Unicode Python 0-based/end-exclusive; output incomplete tidak diisi otomatis.
Contoh di bawah seluruhnya sintetis.

In [7]:
synthetic_records = []
for review in synthetic_reviews:
    records = parse_inference(json.dumps(synthetic_responses[review['review_id']]),
        review, VALIDATOR, 'ai:mock')
    synthetic_records.extend(records)
print({'synthetic_reviews_validated': len(synthetic_reviews),
    'canonical_decisions': len(synthetic_records),
    'mixed_pairs': sum(r['mixed_sentiment'] for r in synthetic_records),
    'uncertain_pairs': sum(r['annotation_status'] == 'uncertain' for r in synthetic_records)})
assert len(synthetic_records) == len(synthetic_reviews) * 6
assert not any(r['training_eligible'] for r in synthetic_records)

{'synthetic_reviews_validated': 17, 'canonical_decisions': 102, 'mixed_pairs': 1, 'uncertain_pairs': 1}


## 8. Gemini Prompt Design

Prompt v1 dihasilkan dari definisi/inclusion/exclusion/overlap taxonomy dan
pedoman interpretasi Stage 3. Instruksi review selalu dianggap data tidak
tepercaya. Prompt meminta enam keputusan structured JSON dengan evidence; bukan
label global dari rating. Manifest mengikat hash prompt, taxonomy, dan guideline.
Revisi prompt memakai versi dan file baru, tidak menimpa versi eksperimen lama.

In [8]:
system_prompt = frozen_prompt(ROOT, CONFIG)
prompt_manifest = read_json(ROOT / 'prompts/absa_gemini_v1.manifest.json')
print({'version': CONFIG['execution']['prompt_version'], 'prompt_characters': len(system_prompt),
    'prompt_sha256': prompt_manifest['prompt_sha256'], 'taxonomy_version': VALIDATOR.version})
assert 'data tidak tepercaya' in system_prompt
assert 'Rating' in system_prompt or 'rating' in system_prompt

{'version': 'absa_gemini_v1', 'prompt_characters': 14056, 'prompt_sha256': '8fa01d2d41125e482bcd38e3fdca363b842c7262b9eb3e8599702956416da770', 'taxonomy_version': '0.1.0-draft'}


## 9. Synthetic Annotation Tests

Fixture: positif, netral, negatif, multi-aspek, no-aspect, mixed, implicit,
sarkasme, negasi, slang, code-switching, typo, overlap toilet, out-of-taxonomy,
Unicode, prompt injection, uncertainty. Ini validasi fixture yang telah ditulis
manual, **bukan hasil inferensi atau akurasi Gemini**.

Tambahan negative tests menolak invalid JSON, aspek hilang dan evidence halusinasi.
Checkpoint/resume diuji di direktori sementara; Run All berikutnya tetap aman.

In [9]:
with tempfile.TemporaryDirectory(prefix='stage4-notebook-', dir=ROOT / '.venv') as temp:
    provider = MockProvider(synthetic_responses)
    pipeline = AnnotationPipeline(provider, VALIDATOR, system_prompt,
        {'origin':'SYNTHETIC_ONLY','prompt_sha256':digest(system_prompt.encode())},
        Path(temp), sleep=lambda _: None)
    mock_report = pipeline.run(synthetic_reviews, execute=True, batch_size=3)
    resume_report = pipeline.run(list(reversed(synthetic_reviews)), execute=True, resume=True)
    synthetic_archives = checkpoint_records(Path(temp), valid_only=False)
    synthetic_queue = quality_queue(synthetic_archives)
print({'synthetic_mock_run': mock_report, 'resume_additional_requests': resume_report['requests'],
    'mock_calls_total': provider.calls})
assert mock_report['valid'] == len(synthetic_reviews) and resume_report['requests'] == 0

{'synthetic_mock_run': {'status': 'COMPLETED', 'planned_reviews': 17, 'requests': 17, 'valid': 17, 'invalid': 0, 'held': 0, 'skipped': 0, 'interrupted': 0, 'unresolved_existing': 0, 'stage4d_complete': False, 'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'usage_missing_requests': 0, 'run_id': 'b777644703e980209ade630e1ef885cd66c73a5462798dd46dc894353a317de8'}, 'resume_additional_requests': 0, 'mock_calls_total': 17}


In [10]:
example_review = next(r for r in synthetic_reviews if r['review_id'] == 'SYN4_MULTI')
negative_test_results = []
for case in ['invalid_json', 'missing_aspect', 'hallucinated_evidence']:
    obj = copy.deepcopy(synthetic_responses[example_review['review_id']])
    if case == 'missing_aspect':
        obj['decisions'].pop()
    if case == 'hallucinated_evidence':
        obj['decisions'][0]['evidence'][0]['quote'] = 'BUKTI TIDAK ADA'
    raw = 'not-json' if case == 'invalid_json' else json.dumps(obj)
    try:
        parse_inference(raw, example_review, VALIDATOR, 'ai:mock')
    except AnnotationError as exc:
        negative_test_results.append({'case':case,'status':'REJECTED_AS_EXPECTED','reason':str(exc)})
    else:
        raise AssertionError('Invalid fixture unexpectedly accepted')
display(pd.DataFrame(negative_test_results))

,case,status,reason
0,invalid_json,REJECTED_AS_EXPECTED,INVALID_JSON
1,missing_aspect,REJECTED_AS_EXPECTED,INCOMPLETE_ASPECT_DECISIONS
2,hallucinated_evidence,REJECTED_AS_EXPECTED,EVIDENCE_MISMATCH


## 10. Real-Data Smoke Test Controls

Default live smoke maksimal lima review dari **calibration**, sehingga locked
holdout tidak dipakai debugging. API key, project Free Tier/kuota, pemeriksaan
privasi, dan `--execute` wajib sebelum upload nyata. Run All tidak melakukan
smoke test nyata. Lima respons valid tidak menjadi ukuran akurasi.

In [11]:
smoke_status = {'status':'NOT_RUN','reason':'Live execution disabled; key/project quota not verified',
    'real_reviews_sent':0,'cloud_generation_requests':0,'cloud_tokens':0}
print(smoke_status)
print('python scripts/run_annotation.py --provider gemini --mode smoke-test --limit 5 --dry-run')
# Eksekusi nyata dilakukan terpisah lewat CLI, bukan dengan mengubah sel Run All.

{'status': 'NOT_RUN', 'reason': 'Live execution disabled; key/project quota not verified', 'real_reviews_sent': 0, 'cloud_generation_requests': 0, 'cloud_tokens': 0}
python scripts/run_annotation.py --provider gemini --mode smoke-test --limit 5 --dry-run


## 11. Inter-Annotator Agreement

IAA hanya pada anotasi dua manusia independen, sebelum adjudikasi. Laporan
memuat raw/Kappa, confusion, support, disagreement dan abstention. Presence
uncertain dikeluarkan denominator biner; mixed/uncertain tidak menjadi kelas
sentimen baru. Kasus Kappa undefined dilaporkan null dan alasan.

In [12]:
human_directory = ROOT / 'data/interim/human_annotations/submissions'
human_a_path = human_directory / 'A_v1.json'
human_b_path = human_directory / 'B_v1.json'
human_a = read_json(human_a_path) if human_a_path.exists() else None
human_b = read_json(human_b_path) if human_b_path.exists() else None
human_agreement = inter_annotator_agreement(human_a, human_b, VALIDATOR)
print({'human_a_available': human_a is not None, 'human_b_available': human_b is not None,
    'agreement_status': human_agreement['status']})
if human_agreement['status'] != 'NOT_COMPUTED':
    display(human_agreement)

{'human_a_available': False, 'human_b_available': False, 'agreement_status': 'NOT_COMPUTED'}


## 12. Gold Standard Preparation

Gold hanya dari original manusia tervalidasi, disagreement diadjudikasi, dan QA.
Semua enam keputusan harus terselesaikan. Mixed completed tetap diarsipkan;
training tiga kelas mengecualikannya. Original manusia dan revisi immutable.
AI tidak diizinkan menjadi gold. Belum ada label nyata yang dibuat notebook ini.

In [13]:
gold_path = ROOT / 'data/processed/human_gold_annotations.jsonl'
human_gold = load_gold(gold_path) if gold_path.exists() else None
print({'human_gold_status':'AVAILABLE' if human_gold else 'NOT_AVAILABLE',
    'gold_reviews':len({row['record']['review_id'] for row in human_gold['rows']}) if human_gold else None,
    'human_annotations_fabricated':False})

{'human_gold_status': 'NOT_AVAILABLE', 'gold_reviews': None, 'human_annotations_fabricated': False}


## 13. Gemini vs Human Evaluation

Utilitas menyediakan detection micro/macro/per-aspect, sentiment accuracy/macro-F1,
per-class dan per-aspect, confusion, end-to-end pair P/R/F1. Missed aspect dihitung
salah, tidak dihilangkan dari denominator sentiment gold-mentioned. Conditional
adalah gold-conditioned scoring, bukan klaim forced-aspect experiment. Invalid,
mixed, abstention dan output hilang dilaporkan. Tidak ada angka penelitian tanpa gold.

In [14]:
saved_evaluation_path = ROOT / CONFIG['full']['pilot_evaluation_path']
ai_vs_human = read_json(saved_evaluation_path) if saved_evaluation_path.exists() else {
    'status':'NOT_COMPUTED','reason':'HUMAN_GOLD_AND_AI_EXPERIMENT_NOT_AVAILABLE'}
print({'ai_vs_human_status':ai_vs_human['status'],
    'real_accuracy_claimed':False if not saved_evaluation_path.exists() else 'See validated experiment report'})
assert evaluate_predictions(None, [], VALIDATOR)['status'] == 'NOT_COMPUTED' 

{'ai_vs_human_status': 'NOT_COMPUTED', 'real_accuracy_claimed': False}


## 14. Ollama Comparison (Optional)

Adapter menerima structured schema yang sama dan hanya mengizinkan endpoint
loopback. Pemeriksaan lingkungan sebelumnya menemukan `qwen2.5:3b` terpasang;
tidak ada download model. Run All membaca snapshot lingkungan lokal dan tidak
memanggil inference Ollama. Pembandingan akurasi menunggu human gold.

In [15]:
environment_path = ROOT / 'reports/stage4_environment.json'
environment_snapshot = read_json(environment_path) if environment_path.exists() else {}
print({'snapshot_checked_on':environment_snapshot.get('checked_on'),
    'ollama_snapshot':environment_snapshot.get('ollama'),
    'comparison_with_human_gold':'NOT_RUN'})

{'snapshot_checked_on': '2026-10-26', 'ollama_snapshot': {'model': 'qwen2.5:3b', 'accessible': True, 'installed_models': ['qwen2.5:3b']}, 'comparison_with_human_gold': 'NOT_RUN'}


## 15. Error Analysis

Human-review queue memprioritaskan invalid/schema/evidence, mixed/uncertain,
rare aspects, implicit, kelas negatif, no-keyword pilot dan potensi duplikasi.
Random audit juga mengambil output yang tampak valid. Validator bukan ukuran
akurasi semantik. Di bawah hanya ringkasan queue **sintetis**.

In [16]:
display(pd.DataFrame([{'origin':'SYNTHETIC_ONLY', 'review_id':row['review_id'],
    'reasons':', '.join(row['reasons']), 'human_review_status':row['human_review_status']}
    for row in synthetic_queue]))
print({'real_error_analysis':'NOT_RUN','synthetic_negative_cases_rejected':len(negative_test_results)})

,origin,review_id,reasons,human_review_status
0,SYNTHETIC_ONLY,SYN4_NEGATIVE,"negative_minority, random_audit, rare_aspect",PENDING
1,SYNTHETIC_ONLY,SYN4_MIXED,mixed,PENDING
2,SYNTHETIC_ONLY,SYN4_IMPLICIT,"implicit, negative_minority",PENDING
3,SYNTHETIC_ONLY,SYN4_NEUTRAL,rare_aspect,PENDING
4,SYNTHETIC_ONLY,SYN4_UNCERTAIN,"needs_context, uncertain",PENDING
5,SYNTHETIC_ONLY,SYN4_MULTI,"negative_minority, rare_aspect",PENDING
6,SYNTHETIC_ONLY,SYN4_NO_ASPECT,general_praise,PENDING
7,SYNTHETIC_ONLY,SYN4_MISSPELLING,misspelling,PENDING
8,SYNTHETIC_ONLY,SYN4_TOILET_MISSING,"negative_minority, rare_aspect",PENDING
9,SYNTHETIC_ONLY,SYN4_NEGATION,negative_minority,PENDING


{'real_error_analysis': 'NOT_RUN', 'synthetic_negative_cases_rejected': 3}


## 16. Full Labeling Execution Controls

Full memerlukan gold nyata, evaluasi locked memenuhi kriteria yang dibekukan,
fingerprint eksperimen cocok, scope wilayah final, status kuota/Free Tier dan
privasi jelas, pemeriksaan sumber, serta persetujuan eksplisit. Status default
menahan full. Tidak ada loop 3.582 review nyata dalam notebook.

Requests dicatat sebelum dikirim, retry terbatas dan 429 berhenti. Resume tidak
mengirim ulang valid output. Crash dengan delivery tidak diketahui ditahan.

In [17]:
try:
    check_live_gates(CONFIG, 'gemini', 'full', reviews[:1], execute=False)
except AnnotationError as exc:
    full_control_status = str(exc)
print({'full_status':'NOT_RUN','default_gate':full_control_status,
    'explicit_approval':CONFIG['full']['explicitly_approved'],
    'criteria_frozen':CONFIG['quality']['criteria_frozen'],
    'geography_finalized':CONFIG['full']['scope_finalized']})

{'full_status': 'NOT_RUN', 'default_gate': 'LIVE_EXECUTION_NOT_ENABLED', 'explicit_approval': False, 'criteria_frozen': False, 'geography_finalized': False}


## 17. Label Export Design

Presence: satu baris/review dengan enam label 1/0/nullable. Sentiment: long format,
termasuk mixed/uncertain dengan training_eligible=false; archive menyimpan semua
evidence/provenance/koreksi. Human gold selalu didahulukan dari reviewed AI.
Tidak ada file dataset produksi berlabel tanpa manusia/QA. Semua pasangan dari
review dan kelompok duplikat tetap satu split. Train/test classifier final
merupakan Stage 5/6. Label AI tidak menjadi ground truth independen classifier.

In [18]:
export_contract = {
    'aspect_presence_dataset.csv':'review_id, coffee_shop_id, review_text, six nullable aspects, label_source, split, geography_status',
    'aspect_sentiment_dataset.csv':'review_id, coffee_shop_id, review_text, aspect_id, sentiment, annotation_source, taxonomy_version, review_flag, adjudication_status, training_eligible, split, geography_status',
    'annotation_archive.jsonl':'canonical record + evidence + source/model/prompt + mixed/uncertain + corrections + split/group',
}
display(pd.DataFrame(export_contract.items(), columns=['output','contract']))
print({'real_export_created_by_notebook':False,'mixed_dropped':False,'uncertain_to_zero':False})

,output,contract
0,aspect_presence_dataset.csv,"review_id, coffee_shop_id, review_text, six nu..."
1,aspect_sentiment_dataset.csv,"review_id, coffee_shop_id, review_text, aspect..."
2,annotation_archive.jsonl,canonical record + evidence + source/model/pro...


{'real_export_created_by_notebook': False, 'mixed_dropped': False, 'uncertain_to_zero': False}


## 18. Quality Report

Status berikut membedakan implementasi dan eksperimen nyata. Synthetic fixtures
memeriksa struktur/aturan pipeline; tidak membuktikan mutu anotasi model.
Laporan utama: `reports/stage4_annotation_report.md`.

In [19]:
quality_report = {
    'stage':4,'taxonomy_version':VALIDATOR.version,
    'stage4a':'IMPLEMENTED_SYNTHETIC_VALIDATED',
    'stage4b':'HUMAN_WORK_PENDING' if human_gold is None else 'CHECK_HUMAN_GOLD_ARTIFACTS',
    'stage4c':ai_vs_human['status'],'stage4d':'NOT_RUN',
    'sdk_google_genai':importlib.metadata.version('google-genai'),
    'primary_candidate_model':CONFIG['model'],
    'api_key_present':setup_status['api_key_present'],
    'project_free_tier':'NOT_VERIFIED',
    'authorization':'OWNER_ATTESTED','geography_unverified_shops':40,
    'pilot_preparation':pilot_preparation,
    'synthetic_reviews':len(synthetic_reviews),'synthetic_decisions':len(synthetic_records),
    'mock_successful_reviews':mock_report['valid'],
    'resume_additional_requests':resume_report['requests'],
    'negative_tests_rejected':len(negative_test_results),
    'cloud_generation_requests_in_notebook':0,'real_reviews_sent_in_notebook':0,
    'raw_sha256':raw_hash_before,
}
assert file_hash(ROOT / 'data/raw/google_maps_reviews_jatinangor_raw.csv') == raw_hash_before
atomic_json(ROOT / 'reports/stage4_notebook_summary.json', quality_report)
display(pd.DataFrame([{'item':k,'status':str(v)} for k,v in quality_report.items() if not isinstance(v,dict)]))

,item,status
0,stage,4
1,taxonomy_version,0.1.0-draft
2,stage4a,IMPLEMENTED_SYNTHETIC_VALIDATED
3,stage4b,HUMAN_WORK_PENDING
4,stage4c,NOT_COMPUTED
5,stage4d,NOT_RUN
6,sdk_google_genai,2.33.0
7,primary_candidate_model,gemini-3.5-flash-lite
8,api_key_present,False
9,project_free_tier,NOT_VERIFIED


## 19. Stage 5 Readiness

Stage 5 **belum siap**: tim perlu mengisi paket manusia, mengukur IAA, melakukan
adjudikasi/QA, membentuk gold, menguji Gemini, memeriksa mutu/biaya/scope, lalu
melabeli dan mengaudit data sesuai protokol. Jangan membuat TF-IDF atau training
SVM/RF/Naive Bayes pada tahap ini.

Langkah pertama tim: buka paket `data/interim/human_annotations/assignments/`,
baca pedoman, kerjakan kalibrasi. Setup key dan project AI Studio dapat berjalan
paralel tanpa mengubah label manusia. Perintah rinci ada pada protokol Stage 4.

In [20]:
print({'stage5_ready':False, 'next_human_step':'50-review calibration',
    'next_api_step':'Set local GEMINI_API_KEY and verify project Free Tier/RPM/RPD',
    'model_training_performed':False})

{'stage5_ready': False, 'next_human_step': '50-review calibration', 'next_api_step': 'Set local GEMINI_API_KEY and verify project Free Tier/RPM/RPD', 'model_training_performed': False}
